In [ ]:
from google.colab import files

uploaded = files.upload()

Saving time_series_60min_singleindex.csv to time_series_60min_singleindex.csv


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('time_series_60min_singleindex.csv')

italy = pd.DataFrame()

italy['date'] = pd.to_datetime(df['utc_timestamp'])
italy = italy.set_index('date')

zones = [
    'IT_NORD_price_day_ahead',
    'IT_CNOR_price_day_ahead',
    'IT_CSUD_price_day_ahead',
    'IT_SUD_price_day_ahead',
    'IT_SARD_price_day_ahead',
    'IT_SICI_price_day_ahead'
]

italy = df[zones]

print(italy.shape)
print(italy.isna().sum())

(50401, 6)
IT_NORD_price_day_ahead    136
IT_CNOR_price_day_ahead    134
IT_CSUD_price_day_ahead    134
IT_SUD_price_day_ahead     134
IT_SARD_price_day_ahead    134
IT_SICI_price_day_ahead    134
dtype: int64


In [3]:
italy = italy.ffill()
italy = italy.dropna()

print(italy.shape)
italy.head()

(50304, 6)


,IT_NORD_price_day_ahead,IT_CNOR_price_day_ahead,IT_CSUD_price_day_ahead,IT_SUD_price_day_ahead,IT_SARD_price_day_ahead,IT_SICI_price_day_ahead
97,39.10,39.10,39.10,39.10,39.10,39.10
98,35.21,35.21,35.21,35.21,35.21,35.21
99,32.43,32.43,32.43,32.43,32.43,32.43
100,33.15,33.15,33.15,33.15,33.15,33.15
101,38.85,38.85,38.85,38.85,38.85,38.85


In [4]:
#regime features
returns = np.log(italy/italy.shift(1))

volatility = returns.rolling(24).std()

features = pd.DataFrame()

features['mean_price'] = italy.mean(axis=1)
features['std_price'] = italy.std(axis=1)
features['mean_return'] = returns.mean(axis=1)
features['volatility'] = volatility.mean(axis=1)
features['max_price'] = italy.max(axis=1)
features['min_price'] = italy.min(axis=1)
features['spread'] = italy.max(axis=1)-italy.min(axis=1)

features = features.dropna()

print(features.shape)
features.head()

/usr/local/lib/python3.12/dist-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)


(50205, 7)


,mean_price,std_price,mean_return,volatility,max_price,min_price,spread
121,45.21,0.000000e+00,-0.141525,0.104435,45.21,45.21,0.0
122,38.80,7.783606e-15,-0.152898,0.107089,38.80,38.80,0.0
123,35.00,0.000000e+00,-0.103072,0.107902,35.00,35.00,0.0
124,33.43,0.000000e+00,-0.045894,0.108278,33.43,33.43,0.0
125,37.58,7.783606e-15,0.117018,0.105933,37.58,37.58,0.0


In [5]:
#returns cleaning
returns = np.log((italy+1)/(italy.shift(1)+1))

volatility = returns.rolling(24).std()

features = pd.DataFrame()

features['mean_price'] = italy.mean(axis=1)
features['std_price'] = italy.std(axis=1)
features['mean_return'] = returns.mean(axis=1)
features['volatility'] = volatility.mean(axis=1)
features['max_price'] = italy.max(axis=1)
features['min_price'] = italy.min(axis=1)
features['spread'] = italy.max(axis=1)-italy.min(axis=1)

features = features.replace(
    [np.inf,-np.inf],
    np.nan
)

features = features.dropna()

print(features.describe())
print(features.shape)

         mean_price     std_price   mean_return    volatility     max_price  \
count  50280.000000  5.028000e+04  50280.000000  50280.000000  50280.000000   
mean      50.720680  5.912877e+00     -0.000006      0.124338     60.418927   
std       15.960749  6.882011e+00      0.120363      0.063926     23.450776   
min        0.000000  0.000000e+00     -1.876218      0.000000      0.000000   
25%       40.270000  7.783606e-15     -0.060374      0.087925     44.390000   
50%       49.661667  3.940115e+00     -0.005916      0.108557     58.090000   
75%       59.978750  8.999064e+00      0.054775      0.138088     71.250000   
max      170.000000  8.737738e+01      2.507267      0.734733    259.030000   

          min_price        spread  
count  50280.000000  50280.000000  
mean      46.039932     14.378994  
std       15.607530     16.878260  
min        0.000000      0.000000  
25%       36.840000      0.000000  
50%       45.540000      9.440000  
75%       55.050000     21.780000  


Average Price
mean_price = 50.7 €/MWh

Αυτό σημαίνει ότι κατά μέσο όρο η ιταλική αγορά είναι αρκετά ακριβότερη από πολλές βόρειες ευρωπαϊκές αγορές.

2. Market Fragmentation
std_price = 5.9 €/MWh
spread = 14.4 €/MWh

Αυτό είναι πολύ σημαντικό.

Ουσιαστικά βλέπουμε ότι:

οι ιταλικές ζώνες δεν κινούνται πάντα μαζί.

Δηλαδή υπάρχουν:

congestion regimes,
transmission bottlenecks,
regional pricing effects.

Αυτό ακριβώς θέλουμε να ανακαλύψει το clustering.

3. Volatility
volatility = 0.124

Η αγορά παρουσιάζει:

low volatility periods,
high volatility periods,
extreme stress periods.
4. Extreme Events

Βλέπουμε:

max spread = 214 €/MWh

Αυτό είναι τεράστιο.

Δηλαδή υπάρχουν ώρες όπου:

κάποιες περιοχές της Ιταλίας αποσυνδέονται σχεδόν πλήρως από τις υπόλοιπες.

Αυτές οι ώρες πιθανότατα θα αποτελέσουν ξεχωριστό market regime.

In [6]:
#Standardization
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X = scaler.fit_transform(features)

In [7]:
#PCA
from sklearn.decomposition import PCA

pca = PCA()

pca.fit(X)

explained = pca.explained_variance_ratio_

for i,v in enumerate(explained):
    print(i+1, round(v,4))


1 0.4855
2 0.25
3 0.1474
4 0.1088
5 0.0069
6 0.0014
7 0.0


In [8]:
import numpy as np

cum_var = np.cumsum(explained)

for i,v in enumerate(cum_var):
    print(i+1, round(v,4))

1 0.4855
2 0.7355
3 0.8829
4 0.9917
5 0.9986
6 1.0
7 1.0


PC1 (48.6%)

Πιθανότατα περιγράφει:

Overall market price level

δηλαδή:

υψηλές τιμές
χαμηλές τιμές
γενική κατάσταση αγοράς
PC2 (25%)

Πιθανότατα περιγράφει:

Regional congestion effects

δηλαδή:

north-south congestion
market splitting
PC3 (14.7%)

Πιθανότατα:

Volatility regime

δηλαδή:

calm market
stressed market
PC4 (10.9%)

Πιθανότατα:

Extreme events regime

In [9]:
#Elbow Method
from sklearn.cluster import KMeans

inertia = []

for k in range(2,11):

    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    km.fit(X)

    inertia.append(km.inertia_)

for i,v in enumerate(inertia):
    print(i+2, v)

2 245810.4269652946
3 201189.91034066115
4 176805.41955746015
5 158727.3268163775
6 143841.25896105435
7 133771.96049566424
8 123978.43833626455
9 116587.59078284011
10 109399.54184325271


In [128]:
#Silhouette Score
from sklearn.metrics import silhouette_score

for k in range(2,11):

    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = km.fit_predict(X)

    score = silhouette_score(X, labels)

    print(k, score)

2 0.35902202199687916
3 0.23775573820548862
4 0.25045081995216095
5 0.24384805016035577
6 0.22571523514553235
7 0.22746899853241354
8 0.22955187026386145
9 0.23511077472144365
10 0.22144981471896866


In [10]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=20
)

features['cluster'] = kmeans.fit_predict(X)

print(
    features.groupby('cluster').mean()
)

         mean_price  std_price  mean_return  volatility  max_price  min_price  \
cluster                                                                         
0         40.772360   3.167684    -0.018037    0.113418  45.729468  38.046987   
1         64.710252  17.590384     0.003195    0.115024  96.107240  52.769268   
2         33.248426   6.073138     0.010680    0.273446  40.056223  25.802523   
3         62.869686   3.445183     0.022740    0.104659  68.323077  60.087718   

            spread  
cluster             
0         7.682481  
1        43.337972  
2        14.253700  
3         8.235359  


In [11]:
print(
    features['cluster']
    .value_counts()
    .sort_index()
)

cluster
0    22768
1     8445
2     4162
3    14905
Name: count, dtype: int64


Regime 0
Normal Market Regime
Χαρακτηριστικά
χαμηλή τιμή
χαμηλό volatility
μικρό spread
μεγαλύτερη συχνότητα εμφάνισης
Ερμηνεία

Αυτό είναι το κανονικό market equilibrium.

Regime 1
Congestion Regime
Χαρακτηριστικά
πολύ υψηλή τιμή
τεράστιο spread
μεγάλη διαφοροποίηση μεταξύ περιοχών
spread = 43 €/MWh
Ερμηνεία

Αυτό είναι σχεδόν σίγουρα:

transmission congestion,
market splitting,
regional scarcity.
Regime 2
High Volatility Regime
Χαρακτηριστικά
volatility = 0.273

δηλαδή υπερδιπλάσια από τα υπόλοιπα regimes.

Ερμηνεία

Αυτό είναι το:

stress regime,
renewable shock regime,
crisis regime.
Regime 3
High Price Stable Regime
Χαρακτηριστικά
υψηλές τιμές
χαμηλό volatility
μικρό spread
Ερμηνεία

Αυτό αντιστοιχεί σε:

peak demand,
high fuel costs,
winter periods.

In [12]:
#Hierarchical Clustering
sample = features.sample(
    n=5000,
    random_state=42
)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_sample = scaler.fit_transform(
    sample.drop(
        columns=['cluster'],
        errors='ignore'
    )
)

from scipy.cluster.hierarchy import linkage

Z = linkage(
    X_sample,
    method='ward'
)

print(Z[:10])

[[2.30100000e+03 2.71600000e+03 0.00000000e+00 2.00000000e+00]
 [7.71000000e+02 3.51900000e+03 3.94213250e-03 2.00000000e+00]
 [1.09500000e+03 3.02200000e+03 1.70836777e-02 2.00000000e+00]
 [1.16300000e+03 1.27900000e+03 2.31232974e-02 2.00000000e+00]
 [2.92300000e+03 3.10700000e+03 2.39040405e-02 2.00000000e+00]
 [1.77500000e+03 4.83500000e+03 2.82646589e-02 2.00000000e+00]
 [3.85000000e+02 4.29200000e+03 2.91443744e-02 2.00000000e+00]
 [3.17000000e+03 4.32900000e+03 3.10985811e-02 2.00000000e+00]
 [3.56900000e+03 4.82800000e+03 3.39954308e-02 2.00000000e+00]
 [1.92900000e+03 3.56000000e+03 3.48141214e-02 2.00000000e+00]]


Due to the quadratic complexity of hierarchical clustering, the analysis was performed on a representative sample of 5,000 observations.

In [13]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(
    n_components=4,
    random_state=42
)

gmm.fit(X)

features['gmm'] = gmm.predict(X)

print(features.groupby('gmm').mean())
print(features['gmm'].value_counts())

     mean_price  std_price  mean_return  volatility  max_price  min_price  \
gmm                                                                         
0     48.178032   2.248997    -0.003517    0.101592  52.768418  47.259483   
1     52.171391   6.584010    -0.007680    0.109258  62.701614  45.923307   
2     52.784117  10.514347     0.010204    0.163712  74.246281  48.491254   
3     54.154289  11.384126     0.013877    0.182193  63.727279  38.063678   

        spread   cluster  
gmm                       
0     5.508935  1.137295  
1    16.778307  1.177992  
2    25.755027  1.322828  
3    25.663601  1.487353  
gmm
0    22652
1    12568
2     9011
3     6049
Name: count, dtype: int64


| GMM | Mean Price | Volatility | Spread | Observations |
| --- | ---------- | ---------- | ------ | ------------ |
| 0   | 48.2       | 0.102      | 5.5    | 22,652       |
| 1   | 52.2       | 0.109      | 16.8   | 12,568       |
| 2   | 52.8       | 0.164      | 25.8   | 9,011        |
| 3   | 54.2       | 0.182      | 25.7   | 6,049        |


GMM 0 — Stable Market Regime
price      low
volatility low
spread     very low

Αυτό είναι το "κανονικό" market equilibrium.

φυσιολογικές τιμές,
μικρή περιφερειακή απόκλιση,
χαμηλή αβεβαιότητα.
GMM 1 — Moderate Congestion Regime
spread = 16.8 €/MWh

Εδώ αρχίζουν να εμφανίζονται:

transmission bottlenecks,
regional imbalances,
congestion effects.
GMM 2 — Volatile Congestion Regime
volatility = 0.164
spread = 25.8 €/MWh

Αυτό φαίνεται να είναι:

high renewable penetration regime,
stressed transmission conditions,
υψηλή αβεβαιότητα.
GMM 3 — Extreme Stress Regime
volatility = 0.182
spread = 25.7 €/MWh

Αυτό είναι το regime των:

market shocks,
extreme events,
κρίσεων και αστάθειας.
Το πιο ενδιαφέρον εύρημα

Παρατήρησε ότι το GMM δεν βρήκε απλώς:

low price / high price

αλλά βρήκε κυρίως:

low volatility
medium congestion
high congestion
stress regime

The Italian electricity market exhibits a low-dimensional latent factor structure and can be decomposed into four statistically distinct market regimes characterized by congestion, volatility and price dynamics.